In [ ]:
# 1. General Inquiry about Flight Information
# Subject: Inquiry about Flight Availability

# Dear [Airline Name] Customer Service,

# I hope this message finds you well. I am writing to inquire about available flights from [Departure City] to [Destination City] for [Date or Range of Dates]. Could you kindly provide information on flight options, timings, and any potential promotions or discounts?

# Thank you for your time and assistance. I look forward to hearing from you.

# Best regards,
# [Your Full Name]
# [Your Contact Information]



In [ ]:
# 2. Flight Cancellation or Delay
# Subject: Request for Assistance Regarding Flight [Flight Number] Delay/Cancellation

# Dear [Airline Name] Customer Service,

# I was scheduled to fly on Flight [Flight Number] from [Departure City] to [Destination City] on [Date]. Unfortunately, the flight was [delayed/cancelled], and I am seeking assistance in regards to [rebooking my flight/a refund/compensation].

# Please let me know the next steps to resolve this issue. I would appreciate your help in ensuring a smooth resolution.

# Thank you in advance for your support.

# Sincerely,
# [Your Full Name]
# [Your Booking Reference Number]
# [Your Contact Information]



In [5]:
e_1="""
Dear Indigo  Customer Service,

I hope this message finds you well. I am writing to inquire about available flights from [Departure City] to [Destination City] for [Date or Range of Dates]. Could you kindly provide information on flight options, timings, and any potential promotions or discounts?

Thank you for your time and assistance. I look forward to hearing from you.

Best regards,
Samir
999999999"""

In [17]:
from ollama import Client
import json
import re

def analyze_email(text, model_name='llama3.1:8b'):
    """Analyze resume text and return structured JSON data"""
    client = Client(host='http://localhost:11434')
    
    # Structured prompt for JSON output
    prompt = f"""Extract the following information from this email in JSON format:
    {text}
    
    Return JSON with these keys:
    -"sentiment type"(string) either "NEGATIVE", "POSITIVE","NEUTRAL" if highly satisfied when senders is happy then positive provide accordingly responses.
    -"email_summary"(string limit upto 30 words)
    -"email_issues"(string)if any problem arrise frome the sender side In 2_3 words
    -"resolution"(string)within 20 words in straight forward manner.

    
    Format: {{ "key": "value" }} without any additional text."""

    try:
        response = client.chat(model=model_name, messages=[
            {'role': 'user', 'content': prompt}
        ])
        
        # Extract JSON from response
        raw_output = response['message']['content']
        
        # Use regex to find JSON in the response
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            return json.loads(json_str)
        
        return {"error": "No JSON found in response"}
    
    except json.JSONDecodeError:
        print("Error decoding JSON response")
        return {"error": "Invalid JSON format"}
    except Exception as e:
        print(f"Error processing request: {str(e)}")
        return {"error": str(e)}

# Example usage
if __name__ == "__main__":

    
    result = analyze_email(e_1)
    
    print(json.dumps(result, indent=2))

{
  "sentiment_type": "POSITIVE",
  "email_summary": "Request for flight information from [Departure City] to [Destination City]",
  "email_issues": "",
  "resolution": "Provide available flights and promotions"
}


In [29]:
from fastapi import FastAPI
from pydantic import BaseModel
class email_Request(BaseModel):
    email_text:str
app = FastAPI()
@app.post("/api/analyze_text")
def analyze_email_function(request: email_Request):
    analyze_email(request.email_text, model_name='llama3.1:8b')
    return "email analyzed successfully"



In [30]:
# import uvicorn
# if __name__ == "__main__":
#     uvicorn.run("routes:app", host="127.0.0.1", port=8084, reload=True)

In [25]:
e_1="""
Dear Indigo  Customer Service,

I hope this message finds you well. I am writing to inquire about available flights from Delhi to Mumbai for [Date or Range of Dates]. Could you kindly provide information on flight options, timings, and any potential promotions or discounts?

Thank you for your time and assistance. I look forward to hearing from you.

Best regards,
Samir
999999999"""

In [26]:
result = analyze_email(e_1)
print(json.dumps(result, indent=2))

{
  "sentiment_type": "POSITIVE",
  "email_summary": "Inquiring about flight options from Delhi to Mumbai",
  "email_issues": "None",
  "resolution": "Available flights and promotions will be provided"
}


In [19]:
# 3. Lost Luggage
# Subject: Urgent: Lost Luggage Report for Flight [Flight Number]

e_2="""Dear [Airline Name] Customer Service,

I am writing to report that my luggage has not arrived following my flight 6E424 from Delhi to Kolkata on 15 june 2025. My luggage details are as follows:

Baggage Claim Number: PNR12345
Description of Luggage: [Brief Description, e.g., "Black suitcase with red straps"]
Could you please update me on the status of my luggage and any next steps I should take?

Thank you for your prompt attention to this matter.

Best regards,
[Your Full Name]
[Your Booking Reference Number]
[Your Contact Information]"""



In [22]:
result = analyze_email(e_2)
print(json.dumps(result, indent=2))

{
  "sentiment_type": "NEGATIVE",
  "email_summary": "Lost luggage on flight from Delhi to Kolkata.",
  "email_issues": "Missing luggage",
  "resolution": "Update luggage status and provide next steps."
}


In [14]:
# 2. Flight Cancellation or Delay
# Subject: Request for Assistance Regarding Flight [Flight Number] Delay/Cancellation

e_3="""Dear [Airline Name] Customer Service,

I was scheduled to fly on Flight [Flight Number] from [Departure City] to [Destination City] on [Date]. Unfortunately, the flight was [delayed/cancelled], and I am seeking assistance in regards to [rebooking my flight/a refund/compensation].

Please let me know the next steps to resolve this issue. I would appreciate your help in ensuring a smooth resolution.

Thank you in advance for your support.

Sincerely,
[Your Full Name]
[Your Booking Reference Number]
[Your Contact Information]"""


In [27]:
result = analyze_email(e_3)
print(json.dumps(result, indent=2))

{
  "sentiment_type": "NEGATIVE",
  "email_summary": "Flight delayed/cancelled, seeking assistance with rebooking/refund/compensation.",
  "email_issues": "Flight issues",
  "resolution": "Rebook or refund as per airline policy."
}


In [ ]:
#to send email
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Email configuration
sender_email = "your_email@gmail.com"
receiver_email = "recipient@example.com"
app_password = "your_16_digit_app_password"  # Use the generated app password

# Create message
message = MIMEMultipart()
message["From"] = sender_email
message["To"] = receiver_email
message["Subject"] = "Test Email from VS Code"
body = "This email was sent using Python in VS Code!"
message.attach(MIMEText(body, "plain"))

# Send email
try:
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(sender_email, app_password)
        server.sendmail(sender_email, receiver_email, message.as_string())
    print("Email sent successfully!")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# # Create a Python script to fetch emails from your Gmail inbox
# import imaplib
# import email

# # Email configuration
# email_user = "your_email@gmail.com"
# email_pass = "your_16_digit_app_password"

# # Connect to Gmail IMAP server
# mail = imaplib.IMAP4_SSL("imap.gmail.com")
# mail.login(email_user, email_pass)
# mail.select("inbox")

# # Search for emails
# status, messages = mail.search(None, "ALL")
# messages = messages[0].split()

# # Fetch the latest 5 emails
# for num in messages[-5:]:
#     status, data = mail.fetch(num, "(RFC822)")
#     raw_email = data[0][1]
#     msg = email.message_from_bytes(raw_email)
#     print(f"Subject: {msg['Subject']}")
#     print(f"From: {msg['From']}\n")

# mail.close()
# mail.logout()

In [ ]:
from ollama import Client
import json
import re
import asyncio
async def analyze_email(text, model_name='llama3.1:8b'):
    """Analyze resume text and return structured JSON data"""
    client = Client(host='http://localhost:11434')
    
    # Structured prompt for JSON output
    prompt = f"""Extract the following information from this email in JSON format:
    {text}
    
    Return JSON with these keys:
    -"sentiment type"(string) either "NEGATIVE", "POSITIVE","NEUTRAL" if highly satisfied when senders is happy then positive provide accordingly responses.
    -"email_summary"(string limit upto 30 words)
    -"email_issues"(string)if any problem arrise frome the sender side In 2_3 words
    -"resolution"(string)within 20 words in straight forward manner.
    -"suggested_agent_response"(string) what the agent should reply back on this email provide within 50 words

    
    Format: {{ "key": "value" }} without any additional text."""

    try:
        response = client.chat(model=model_name, messages=[
            {'role': 'user', 'content': prompt}
        ])
        
        # Extract JSON from response
        raw_output = response['message']['content']
        
        # Use regex to find JSON in the response
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            return json.loads(json_str)
        
        return {"error": "No JSON found in response"}
    
    except json.JSONDecodeError:
        print("Error decoding JSON response")
        return {"error": "Invalid JSON format"}
    except Exception as e:
        print(f"Error processing request: {str(e)}")
        return {"error": str(e)}

In [32]:
from ollama import Client
import json
import re
import asyncio

def create_response_prompt(email_data):
    """Create a detailed prompt for generating customer-focused responses"""
    return f"""
    Based on this email situation:
    - Summary: {email_data['email_summary']}
    - Issue Type: {email_data['email_issues']}
    - Sentiment: {email_data['sentiment_type']}
    
    Generate a response that:
    1. Addresses the customer appropriately and personally
    2. Prioritizes addressing their main concerns in order of importance
    3. Shows empathy by acknowledging their experience
    4. Includes a sincere apology if there was a company mistake
    5. Takes clear ownership of next steps
    
    The response should be professional, under 100 words, and focus on concrete actions and solutions.
    """

def analyze_email_sync(text, model_name='llama3.1:8b'):
    """Synchronous version of email analysis"""
    client = Client(host='http://localhost:11434')
    
    # First analysis to get basic email information
    initial_prompt = f"""Extract the following information from this email in JSON format:
    {text}
    
    Return JSON with these keys:
    -"sentiment_type"(string) either "NEGATIVE", "POSITIVE","NEUTRAL" if highly satisfied when senders is happy then positive provide accordingly responses.
    -"email_summary"(string limit upto 30 words)
    -"email_issues"(string)if any problem arise from the sender side In 2-3 words
    -"resolution"(string)within 20 words in straight forward manner.
    
    Format: {{ "key": "value" }} without any additional text."""

    try:
        # Get initial analysis
        initial_response = client.chat(model=model_name, messages=[
            {'role': 'user', 'content': initial_prompt}
        ])
        
        # Extract and parse initial JSON
        json_match = re.search(r'\{.*\}', initial_response['message']['content'], re.DOTALL)
        if not json_match:
            return {"error": "No JSON found in response"}
        
        email_data = json.loads(json_match.group())
        
        # Generate enhanced response using the parameters
        response_prompt = create_response_prompt(email_data)
        
        response_generation = client.chat(model=model_name, messages=[
            {'role': 'user', 'content': response_prompt}
        ])
        
        # Add generated response to the results
        email_data['suggested_agent_response'] = response_generation['message']['content'].strip()
        
        # Add quality checks
        email_data['response_quality_checks'] = {
            'proper_address': check_proper_address(email_data['suggested_agent_response']),
            'prioritized_concerns': check_prioritized_concerns(email_data['suggested_agent_response'], email_data['email_issues']),
            'shows_empathy': check_empathy(email_data['suggested_agent_response']),
            'appropriate_apology': check_apology(email_data['suggested_agent_response'], email_data['sentiment_type']),
            'takes_ownership': check_ownership(email_data['suggested_agent_response'])
        }
        
        return email_data
    
    except json.JSONDecodeError:
        return {"error": "Invalid JSON format"}
    except Exception as e:
        return {"error": str(e)}

def check_proper_address(response):
    """Check if response includes proper customer address"""
    address_patterns = ['Dear', 'Hello', 'Hi']
    return any(pattern in response for pattern in address_patterns)

def check_prioritized_concerns(response, issues):
    """Check if main issues are addressed early in the response"""
    return issues.lower() in response.lower()[:len(response)//2]

def check_empathy(response):
    """Check for empathy indicators in response"""
    empathy_phrases = ['understand', 'appreciate', 'apologize', 'sorry', 'concern']
    return any(phrase in response.lower() for phrase in empathy_phrases)

def check_apology(response, sentiment):
    """Check for appropriate apology if sentiment is negative"""
    if sentiment == "NEGATIVE":
        apology_phrases = ['apologize', 'sorry', 'regret']
        return any(phrase in response.lower() for phrase in apology_phrases)
    return True

def check_ownership(response):
    """Check if response takes ownership of next steps"""
    ownership_phrases = ['will', 'shall', 'going to', 'I will', 'we will']
    return any(phrase in response.lower() for phrase in ownership_phrases)

# Example usage
if __name__ == "__main__":
    sample_email = """
    Dear Customer Service,
    My luggage has been missing since my flight from Delhi to Kolkata on 15 June 2025.
    This is causing me significant inconvenience as I have important items in it.
    Please help me locate it as soon as possible.
    Regards,
    John Doe
    """
    
    result = analyze_email_sync(sample_email)
    print(json.dumps(result, indent=2))

{
  "sentiment_type": "NEGATIVE",
  "email_summary": "Missing luggage on flight from Delhi to Kolkata",
  "email_issues": "Lost luggage",
  "resolution": "Locate and return luggage as soon as possible",
  "suggested_agent_response": "Here is a potential response:\n\nDear [Customer's Name],\n\nI am writing to personally apologize for the distress caused by missing your luggage on flight from Delhi to Kolkata. I understand that this can be very frustrating and inconvenient.\n\nTo resolve this issue, I have escalated the matter to our lost and found team who will work closely with you to locate your luggage as soon as possible. In the meantime, we would like to offer you a reimbursement for essential items or provide temporary replacement clothes if needed. Please let me know how we can assist further.\n\nBest regards,\n[Your Name]",
  "response_quality_checks": {
    "proper_address": true,
    "prioritized_concerns": false,
    "shows_empathy": true,
    "appropriate_apology": true,
   